In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np

from dynamicslib.consts import muEM
from dynamicslib.continuation import find_bif, adaptive_cont
from dynamicslib.common import jacobi_constant, prop
from dynamicslib.targeter import dc_overconstrained
from tqdm.auto import tqdm
from dynamicslib.plotting import plotly_display, bifurcation_search, test_new_family
from dynamicslib.common_targetters import spatial_perpendicular, spatial_period_fixed, fullstate_minus_one, axial

family = "family"

root = os.getcwd()
file = f"{root}/database/{family}.csv"
db = pd.read_csv(file).set_index("Index")

In [ ]:
bifurcation_search(db[:], line_color_by="Jacobi Constant")

In [ ]:
# targetter = fullstate_minus_one(1, 4, 0.0, 1e-14, muEM)
targetter = spatial_perpendicular(1e-13, muEM)
# targetter = axial(1e-13, muEM)


func = targetter.f_df_stm

In [ ]:
keys = ["Initial x", "Initial y", "Initial z", "Initial vx", "Initial vy", "Initial vz"]
keys += ["Period"]
for key in keys:
    if key not in db.columns:
        db[key] = 0.0


im1 = 16
im0 = 17

In [ ]:
# %matplotlib ipympl
Xm0 = targetter.get_X(db[keys[:-1]].iloc[im0].values, db["Period"].iloc[im0])
Xm1 = targetter.get_X(db[keys[:-1]].iloc[im1].values, db["Period"].iloc[im1])
x0, tf = db[keys[:-1]].iloc[im1].values, db["Period"].iloc[im1]


# Xm0 = targetter.get_X(prop(db[keys[:-1]].iloc[im0].values, db["Period"].iloc[im0] / 2, int_tol=1e-10)[:, -1], db["Period"].iloc[im0])
# Xm1 = targetter.get_X(prop(db[keys[:-1]].iloc[im1].values, db["Period"].iloc[im1] / 2, int_tol=1e-10)[:, -1], db["Period"].iloc[im1])
Xm1, _, _ = dc_overconstrained(Xm1, func, 1e-10, debug=True)
Xm0, _, _ = dc_overconstrained(Xm0, func, 1e-10, debug=True)
f, df, stm = func(Xm1)
svd = np.linalg.svd(df)
tangent = svd.Vh[-1]
sprev = np.linalg.norm(Xm0 - Xm1)
if np.dot(tangent, Xm0 - Xm1) / sprev < 0:
    tangent *= -1
sprev = np.linalg.norm(Xm0 - Xm1)
print(f"Last dist is {sprev:.3e}")

In [ ]:
X0, tan1 = find_bif(
    Xm1,
    func,
    tangent,
    sprev / 5,
    targ_tol=1e-12,
    bisect_tol=1e-10,
    bif_type="tangent",
    debug=True,
    scale=4,
)

In [ ]:
X1 = Xm0.copy()
# X1[-1] *= 3
# X2, df, _ = dc_overconstrained(X1, func, 1e-12, debug=True)
f, df, stm = func(X1)
tangent = np.linalg.svd(df).Vh[-2]
tangent

In [ ]:
Xsp1, esp1, (DFs, tangents, s_vals) = adaptive_cont(
    X0,
    func,
    tangent,
    1e-3,
    1e-6,
    30.0,
    1e-13,
    exact_tangent=True,
    max_iter=10,
    target_iter=5,
    pseudo=False,
)

In [ ]:
_, dF, _ = dc_overconstrained(Xsp1[-1], func, 1e-12)
tangent = np.linalg.svd(dF).Vh[-1]
if np.dot(Xsp1[-1] - Xsp1[-2], tangent) < 0:
    tangent *= -1
Xsp2, esp2, _ = adaptive_cont(
    Xsp1[-1],
    func,
    tangent,
    1.5e-2,
    1e-5,
    5,
    1e-13,
    max_iter=10,
    rate=1.05,
    reduce_maxiter=1.3,
    target_iter=5,
    exact_tangent=True,
    pseudo=True,
)

In [ ]:
Xsp = [*Xsp1, *Xsp2[1:]]
esp = [*esp1, *esp2[1:]]
Xsm = []
esm = []

param_names = ["Index"]
param_names += [f"Initial {v}" for v in ["x", "y", "z", "vx", "vy", "vz"]]
param_names += ["Period", "Jacobi Constant", "Stability Index"]
param_names += [f"Eig{n+1}" for n in range(6)]


xyzs = []
data = []


iterator = list(enumerate(zip(Xsm, esm)))[::-1]
for i, (X, evals) in tqdm(iterator):
    Xcp = X.copy()
    x0 = targetter.get_x0(Xcp)
    tf = targetter.get_period(Xcp)
    xyzs.append(prop(x0, tf, muEM, 1e-10))
    args = np.argsort(np.abs(evals))
    stabind = np.max([(np.abs(lam) + 1 / np.abs(lam)) / 2 for lam in evals])
    jc = jacobi_constant(x0)
    data.append([-i - 1, *x0, tf, jc, stabind, *evals])

iterator = list(enumerate(zip(Xsp, esp)))
for i, (X, evals) in tqdm(iterator):
    Xcp = X.copy()
    x0 = targetter.get_x0(Xcp)
    tf = targetter.get_period(Xcp)
    xyzs.append(prop(x0, tf, muEM, 1e-9))
    args = np.argsort(np.abs(evals))
    stabind = np.max([(np.abs(lam) + 1 / np.abs(lam)) / 2 for lam in evals])
    jc = jacobi_constant(x0)
    data.append([i, *x0, tf, jc, stabind, *evals])

In [ ]:
df = pd.DataFrame(data, columns=param_names).set_index("Index")
for key in param_names[1:]:
    if np.all(df[key].abs() < 1e-8):
        df = df.drop(columns=[key])

df.to_csv("L2 Halo2.csv")

In [ ]:
test_new_family(df)

In [ ]:
a = None
b = None
c = 10
plotly_display(xyzs[a:b:c], df[a:b:c], color_by="Index", port=8010)